# APUSH LEQ Grader SLM — v1 QLoRA train + base-vs-tuned eval (Colab)

Runs the Day-3 gate on a GPU: train the v1 QLoRA adapter, then eval it on both tracks and put the numbers on the board.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough for a 0.5B model).

Order: check GPU → clone → install → (HF token) → train → eval litmus → eval real → print base-vs-tuned table.

`artifacts/data/` is committed to the repo, so no data regeneration is needed — the clone already has `train_chat.jsonl`, `eval_cases.jsonl`, and `eval_real_cases.jsonl`.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'

## 2. Clone the repo

Public repo — the committed `artifacts/data/` comes with it. Re-run-safe: pulls latest if the folder already exists.

In [ ]:
import os
REPO = 'https://github.com/aryanjverma/slm.git'
if not os.path.isdir('/content/slm'):
    !git clone $REPO /content/slm
else:
    !cd /content/slm && git pull
%cd /content/slm
!wc -l artifacts/data/train_chat.jsonl artifacts/data/eval_cases.jsonl artifacts/data/eval_real_cases.jsonl

## 3. Install training extras

Installs the package with the `[train]` extra (unsloth / trl / peft / bitsandbytes). Takes a few minutes; a pip resolver warning about pre-installed Colab packages is normal.

In [ ]:
!pip install -q -e ".[train]"

## 4. (Optional) Hugging Face token

Avoids rate-limited base-model downloads. Add a Colab secret named `HF_TOKEN` (🔑 in the left sidebar), or skip this cell to download anonymously.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('No HF_TOKEN (downloading anonymously):', e)

## 5. Train the v1 QLoRA adapter

Defaults: `Qwen/Qwen2.5-0.5B-Instruct`, 300 steps, LoRA rank 16 — a few minutes on a T4. Writes the adapter to `artifacts/models/apush-frq-grader-v1/`.

In [ ]:
!python scripts/train_qlora.py \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --data artifacts/data/train_chat.jsonl \
  --output artifacts/models/apush-frq-grader-v1 \
  --max-steps 300

## 6. Refresh the deterministic baselines

Regenerates `inflated_prompted_base` and `apush_grader_reference` on the same 198-case litmus set so the tuned model is compared against them on identical inputs. CPU-fast.

In [ ]:
!python -m apush_frq_grader_slm.cli.run_eval \
  --eval-path artifacts/data/eval_cases.jsonl \
  --output-dir artifacts/eval

## 7. Eval the tuned model — litmus track (the gate)

Runs the tuned adapter over the synthetic contract + adversarial set. This is the base-vs-tuned regression signal.

In [ ]:
!python scripts/eval_hf_model.py \
  --model artifacts/models/apush-frq-grader-v1 \
  --model-name apush_frq_grader_v1 \
  --eval-path artifacts/data/eval_cases.jsonl \
  --output-dir artifacts/eval

## 8. Eval the tuned model — real track (College Board agreement)

Row-score agreement and QWK vs the ~72 real CB essays. Eval only — these never entered training.

In [ ]:
!python scripts/eval_hf_model.py \
  --model artifacts/models/apush-frq-grader-v1 \
  --model-name apush_frq_grader_v1 \
  --eval-path artifacts/data/eval_real_cases.jsonl \
  --real-eval \
  --output-dir artifacts/eval

## 9. Base-vs-tuned summary table

Reads every summary written to `artifacts/eval/` and prints the litmus comparison. The gate is met when `apush_frq_grader_v1` beats `inflated_prompted_base` on grounding and robustness.

In [ ]:
import json, glob, os

def load(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Litmus summaries: baselines live in summary.jsonl; tuned model in its own *_summary.jsonl.
rows = {}
sp = 'artifacts/eval/summary.jsonl'
if os.path.exists(sp):
    for r in load(sp):
        rows[r['model_name']] = r
for p in glob.glob('artifacts/eval/*_summary.jsonl'):
    if 'slice' in p or 'real' in p or p.endswith('summary.jsonl') and os.path.basename(p) == 'summary.jsonl':
        continue
    for r in load(p):
        rows[r['model_name']] = r

order = ['inflated_prompted_base', 'apush_frq_grader_v1', 'apush_grader_reference']
ranked = [rows[m] for m in order if m in rows] + [r for m, r in rows.items() if m not in order]

hdr = f"{'model':<28} {'n':>4} {'json':>6} {'rubric':>7} {'ground':>7} {'robust':>7} {'total':>6}"
print('LITMUS (198-case synthetic contract + adversarial)')
print(hdr); print('-' * len(hdr))
for r in ranked:
    print(f"{r['model_name']:<28} {r['count']:>4} {r['structured_output_valid_rate']:>6.2f} "
          f"{r['rubric_accuracy_mean']:>7.2f} {r['evidence_grounding_rate']:>7.2f} "
          f"{r['robustness_mean']:>7.2f} {r['total_score_mean']:>6.2f}")

# Real track (tuned model only).
rp = 'artifacts/eval/apush_frq_grader_v1_real_summary.jsonl'
if os.path.exists(rp):
    r = load(rp)[0]
    print('\nREAL (College Board essays)')
    print(f"  cases={r['count']}  json_valid={r['structured_output_valid_rate']:.2f}")
    print(f"  row exact={r['exact_match_rate']:.2f}  row within-1={r['within_one_rate']:.2f}")
    print(f"  total exact={r['total_exact_match_rate']:.2f}  total within-1={r['total_within_one_rate']:.2f}  QWK={r['qwk']}")

## 10. (Optional) Save the adapter to Google Drive

The adapter is `.gitignore`d and Colab storage is ephemeral. Copy it to Drive to keep it after the runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/slm_models
!cp -r artifacts/models/apush-frq-grader-v1 /content/drive/MyDrive/slm_models/
!ls -la /content/drive/MyDrive/slm_models/apush-frq-grader-v1